In [ ]:
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# 元データのうちbinary以外の値のカラム名

list_non_binary = [
    "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways","Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points", "Cover_Type"
]

# 目的変数
y_col = "Cover_Type"

## 1. データの読み込み

In [ ]:
# # 初回取得時のみ実行
# from ucimlrepo import fetch_ucirepo
# covertype = fetch_ucirepo(id=31) 
# df_data = covertype.data.original.copy()

# # 毎回fetchする手間を省くために、ローカルに保存しておく
# df_data.to_csv("../data/data.csv", index=False, encoding="utf-8")

In [ ]:
# fetchではなく、ローカル保存を読み込む場合

if "df_data" not in locals():
    df_data = pd.read_csv("../data/data.csv")

#　元データが約75MBくらいだが、60万行程度なのでpolarsではなく、pandasで進める
# メモリなどの処理に影響があれば、polarsに切り替える

## 2. データの中身の確認

In [ ]:
df_eda = df_data.copy()

In [ ]:
df_eda.head(5)

In [ ]:
df_eda.shape

In [ ]:
df_eda.dtypes

### 2.1 欠損値確認

In [ ]:
df_eda.isnull().sum()

# →欠損はなし

### 2.2 バイナリのカラムの確認

In [ ]:
df_eda.columns

In [ ]:
# バイナリーカラムの0,1だけが格納されているか確認

for col in df_eda.drop(list_non_binary, axis=1).columns:

    print(col, df_eda[col].unique())

### 2.3 目的変数の確認

In [ ]:
df_eda[y_col].value_counts()

# Cover_Type
# 2    283301
# 1    211840
# 3     35754
# 7     20510
# 6     17367
# 5      9493
# 4      2747
# →目的変数は1~7の整数なので、定義通りの値が格納されている

## 3. EDA

In [ ]:
# バイナリ以外のデータ概要の確認
df_eda[list_non_binary].describe()

# Vertical_Distance_To_Hydrologyのminが-173
# 間違った値なのか、垂直距離をします変数なので、高い方が正、低い方が負、などの正負が分布として存在するのか確認してみる。
# UCIの公式サイトには定義なし。

### 3.1 Vertical_Distance_To_Hydrologyの値域の確認

In [ ]:
sns.histplot(data=df_eda, x="Vertical_Distance_To_Hydrology")
plt.xlim(
    df_eda["Vertical_Distance_To_Hydrology"].min(),
    df_eda["Vertical_Distance_To_Hydrology"].max(),
)
plt.show()

# 負数が分布として存在している。
# 本来なら値域の定義をしっかりと確認するべきだが、uciの公式サイトに記載がなかったため、
# 負数は何かしらのミスで混ざった値ではなく、元からその値として格納された、という前提を本分析ではおく。

### 3.2 離散化

In [ ]:
# EDAでは数値特徴量の分布や目的変数との関係を視覚的に確認しやすくするため、
# 一定幅で離散化している。モデル学習には元の連続値を使用する。

df_eda["Elevation"] = (df_data["Elevation"]//250) *250
df_eda["Aspect"] = (df_eda["Aspect"]//30)*30
df_eda["Slope"] = (df_eda["Slope"]//5)*5
df_eda["Horizontal_Distance_To_Hydrology"] = (df_eda["Horizontal_Distance_To_Hydrology"]//50)*50
df_eda["Vertical_Distance_To_Hydrology"] = (df_eda["Vertical_Distance_To_Hydrology"]//50)*50
df_eda["Horizontal_Distance_To_Roadways"] = (df_eda["Horizontal_Distance_To_Roadways"]//250)*250
df_eda["Hillshade_9am"] = (df_eda["Hillshade_9am"]//10)*10
df_eda["Hillshade_Noon"] = (df_eda["Hillshade_Noon"]//10)*10
df_eda["Hillshade_3pm"] = (df_eda["Hillshade_3pm"] //10)*10
df_eda["Horizontal_Distance_To_Fire_Points"] = (df_eda["Horizontal_Distance_To_Fire_Points"]//250)*250


In [ ]:
# 離散化の確認
df_eda[list_non_binary].describe()

### 3.3 目的変数を軸としたクロス集計

#### 3.3.1 数値データ

In [ ]:
# # グラフ領域の設定
# ncols = 2
# nrows = math.ceil((len(list_non_binary)-1) / ncols)

# fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, 4 * nrows))
# axes = axes.flatten() 

# for i, col in enumerate(list_non_binary):

#     if col == y_col:
#         continue

#     cross_tab = pd.crosstab(df_eda[y_col], df_eda[col])

#     cross_tab.plot(
#         kind="bar", stacked=True, ax=axes[i], colormap="tab10", rot=0
#     )

#     axes[i].set_title(f"{y_col} vs {col}")
#     axes[i].set_xlabel(y_col)
#     axes[i].set_ylabel("Counts")
#     axes[i].legend(title=col, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
#     # axes[i].legend(title=col, fontsize=8, title_fontsize=9)

# # 余った空白のサブプロットがある場合は非表示にする
# for j in range(i + 1, len(axes)):
#   fig.delaxes(axes[j])

# plt.tight_layout()
# plt.show()

In [ ]:
# １つのグラフにまとめると、凡例も異なるので、みづらくなった。
# そのため、１個１個描画する

for col in list_non_binary:

    if col == y_col:
        continue

    plt.figure(figsize=(6, 4))
    # cross_tab = pd.crosstab(df_eda[y_col], df_eda[col])
    cross_tab_pct = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

    cross_tab_pct.plot(
        kind="bar", stacked=True, colormap="viridis"
    )

    plt.title(f"Crosstab of {y_col} and {col}")
    plt.xlabel(y_col)
    plt.ylabel("Counts")
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=5)
    # plt.tight_layout()

    plt.show()


#### 3.3.2　バイナリデータ

In [ ]:
df_eda.drop(list_non_binary, axis=1).columns

In [ ]:
binary_cols = df_eda.drop(list_non_binary, axis=1).columns

# グリッド配置
display_cols = 2
display_rows = (len(binary_cols) + 1) // display_cols

fig, axes = plt.subplots(
    display_rows, display_cols, figsize=(12, 4 * display_rows)
)
axes = axes.flatten()  # 多次元配列を1次元に平坦化してループしやすくする

for i, col in enumerate(binary_cols):
    ct = pd.crosstab(df_eda[y_col], df_eda[col])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="barh", stacked=True, ax=axes[i], color=["#e0e0e0", "#ff7f0e"])
    axes[i].set_title(col)
    axes[i].set_xlabel("%")

# 余った描画エリア（空白のグラフ）を非表示にする
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# →one hot eocodingされてるカラムなので、基本的に０が多い特性がある
# その前提でも、1が多いカラムは、特徴があるといえる可能性が高い
# uciの公式ページにone hotの記載がないので、.sumとかを計算して検算する

In [ ]:
# Wilderness_Areaがone hot encodingになっているか確認
len(df_eda) == df_eda[binary_cols].filter(like="Wilderness_Area").sum().sum()

# →true
# uciの公式ページにはone hot encodingと明記はないが、
# 合計が一致するので、本分析においては、one hot encodingされたカラムとして扱う

In [ ]:
# Soil_Typeがone hot encodingになっているか確認
len(df_eda) == df_eda[binary_cols].filter(like="Soil_Type").sum().sum()

# →true
# uciの公式ページにはone hot encodingと明記はないが、
# 合計が一致するので、本分析においては、one hot encodingされたカラムとして扱う


## READMEのグラフ描画

In [ ]:
# 元データのcover_typesごとのサンプル数をグラフにする

cover_types = ["1", "2", "3", "4", "5", "6", "7"]
samples = [211840, 283301, 35754, 2747, 9493, 17367, 20510]

# 色指定
colors = ["#2b5c8f", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#e6ab02", "#a6761d"]

fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
bars = ax.bar(cover_types, samples, color=colors, width=0.6, edgecolor="none")

# 背景のグリッド設定
ax.set_axisbelow(True)
ax.yaxis.grid(True, color="#f0f0f0", linestyle="-", linewidth=1)

# 不要な外枠（上・右・左）を消す
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color("#cccccc") # 下線だけ薄いグレーで残す

ax.set_title("Sample Distribution by Cover Type", fontsize=16, pad=20, fontweight="bold", color="#333333")
ax.set_xlabel("Cover Type", fontsize=12, labelpad=10, color="#333333")
ax.set_ylabel("Number of Samples", fontsize=12, labelpad=10, color="#333333")

# Y軸のカンマ区切りで表示
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

# 各バーの真上に数値をカンマ区切りで表示
for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:,}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),  # バーの少し上に配置
                textcoords="offset points",
                ha="center", va="bottom", fontsize=10, fontweight="semibold", color="#444444")

# レイアウトの自動調整と表示
plt.tight_layout()
plt.show()